# Matrix Product Operators

In this notebook we explore how to define a Matrix Product Operator using `ITensor`. We will focus, in particular, on the definition of the Hamiltonian operators for the examples discussed in Lecture 3. 

We will start by explicitly building some MPOs for simple (small) systems. Then we will illustrate how to exploit the `ITensorMPS` library to do the "dirty work". At the end of this notebook you should have all the tools needed to implement the MPO of your choice.

## Fundamentals

In [1]:
using Pkg
Pkg.activate("..")
using LinearAlgebra
using ITensors
using ITensorMPS

  Activating project at `~/Library/CloudStorage/OneDrive-UniversitàdegliStudidiMilano/Work/Workspace/Unimi/Corsi/TN_Ulm/Materiale_dispense/TN_Notebooks`


Let's start simple by considering a single spin $1/2$ system, and the generators of the corresponding algebra of operators.

In [24]:
i = Index(2, "site,Spin_1/2,i")
ψ =ITensor(ComplexF64,i)

ITensor ord=1 (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.EmptyStorage{ComplexF64, NDTensors.Dense{ComplexF64, Vector{ComplexF64}}}

In [25]:
pauliMatX = ComplexF64.([0. 1.;1. 0.])
pauliMatY = ComplexF64.([0. -im; im 0.])
pauliMatZ = ComplexF64.([1. 0.;0. -1.])
pauliMatId = ComplexF64.([1. 0.;0. 1.])
pauliMats = [pauliMatX, pauliMatY, pauliMatZ, pauliMatId];

In [26]:
pauliOps = [ITensor(ComplexF64, pauliMat, i', i) for pauliMat in pauliMats];

In [27]:
println(pauliOps[1])

ITensor ord=2
Dim 1: (dim=2|id=821|"Spin_1/2,i,site")'
Dim 2: (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 0.0 + 0.0im  1.0 + 0.0im
 1.0 + 0.0im  0.0 + 0.0im



In the jargon of the last lecture, the tensors `pauliOps[k]`are (trivial) MPOs:

$\sigma_x = \sum_{i,i'=1}^2 O_{i'}^i \ket{i'}\bra{i}$

As we already saw in the first lecture, we can apply an operator to a state $\ket{\psi}$ by contracting 
$$
O \ket{\psi} = \sum_{i',i,k}O_{i'}^i \psi_k \ket{i'} \braket{i|k} = \sum_{i'} \left (\sum_{k} O_{i'}^k \psi_k \right ) \ket{i'}
$$

Once initialized the state to $\ket{\uparrow} = (1,0)^T$

In [28]:
ψ[1] = 1
ψ[2] = 0
@show ψ

ψ = ITensor ord=1
Dim 1: (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
 1.0 + 0.0im
 0.0 + 0.0im



ITensor ord=1 (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

we obtain:

In [31]:
pauliXOp = pauliOps[1]
resultX = noprime(pauliXOp * ψ)
@show resultX

resultX = ITensor ord=1
Dim 1: (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
 0.0 + 0.0im
 1.0 + 0.0im



ITensor ord=1 (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [30]:
pauliYOp = pauliOps[2]
@show pauliYOp
resultY = noprime(pauliYOp * ψ)
@show resultY

pauliYOp = ITensor ord=2
Dim 1: (dim=2|id=821|"Spin_1/2,i,site")'
Dim 2: (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 0.0 + 0.0im  0.0 - 1.0im
 0.0 + 1.0im  0.0 + 0.0im

resultY = ITensor ord=1
Dim 1: (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
 0.0 + 0.0im
 0.0 + 1.0im



ITensor ord=1 (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [32]:
pauliZOp = pauliOps[3]
@show pauliZOp
resultZ = noprime(pauliZOp * ψ)
@show resultZ

pauliZOp = ITensor ord=2
Dim 1: (dim=2|id=821|"Spin_1/2,i,site")'
Dim 2: (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 1.0 + 0.0im   0.0 + 0.0im
 0.0 + 0.0im  -1.0 + 0.0im

resultZ = ITensor ord=1
Dim 1: (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
 1.0 + 0.0im
 0.0 + 0.0im



ITensor ord=1 (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

Let us now introduce another spin $1/2$ system. 

In [47]:
 j= Index(2, "site,Spin_1/2,j")

(dim=2|id=215|"Spin_1/2,j,site")

In [55]:
pauliOps2 = [ITensor(ComplexF64, pauliMat, j', j) for pauliMat in pauliMats];
pauliOpX2,pauliOpY2,pauliOpZ2,pauliOpId2 = pauliOps2;

In [56]:
#For the sake of simplicity we rename the first list of operators referring to
pauliOps1 = pauliOps;
pauliOpX1,pauliOpY1,pauliOpZ1,pauliOpId1 = pauliOps1;

We now define the matrix of the operator $\sigma_1^x \otimes \sigma_2^z$ acting on both spins. Then we derive the MPO representation of the same operator.

In [137]:
twoSiteOp = pauliOpX1 * pauliOpZ2

ITensor ord=4 (dim=2|id=821|"Spin_1/2,i,site")' (dim=2|id=821|"Spin_1/2,i,site") (dim=2|id=215|"Spin_1/2,j,site")' (dim=2|id=215|"Spin_1/2,j,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

By following our notational convention we write the two site operator as:
$$
O = \sum_{\substack{i,j \\ i',j'}} O_{i',j'}^{i,j} \ket{i'j'}\hspace{-3pt}\bra{i,j}
$$

The matrix representation of the operator can be obtained by means of combiners as we did in the first lecture as follows:

In [138]:
c1 = combiner(j,i,tags="combined i,j")
c2 = combiner(j',i',tags="combined i',j'")
appo = c2 * twoSiteOp * c1
matrix(appo)

4×4 Matrix{ComplexF64}:
 0.0+0.0im   0.0+0.0im  1.0+0.0im   0.0+0.0im
 0.0+0.0im   0.0+0.0im  0.0+0.0im  -1.0+0.0im
 1.0+0.0im   0.0+0.0im  0.0+0.0im   0.0+0.0im
 0.0+0.0im  -1.0+0.0im  0.0+0.0im   0.0+0.0im

This is the representation of the operator we would get by operating the tensor product.

In order to build the MPO representation, on the other side, we need to combine the indices differently, since we need to "group" the input-output indices of the same site:
$$
O =\sum_{\substack{i,j \\ i',j'}} O_{i',j'}^{i,j} \ket{i'j'}\hspace{-3pt}\bra{i,j} = 
\sum_{\substack{i,j \\ i',j'}}  O_{i',j'}^{i,j} \ket{i'} \hspace{-3pt} \bra{i} \otimes \ket{j'} \hspace{-3pt} \bra{j} =
\sum_{\substack{i,j \\ i',j'}}  O_{(i',i),(j',j)} \ket{(i',i)} \otimes \ket{(j'j)}
$$

Unfortunately the use of combiners leads to some issues. Let's do it manually.

In [169]:
#The function gets an array of indices and an array of sizes and returns the position in the vectorized version of the tensor corresponding to those indices
function reshapeTama(inds::Vector{<:Integer}, sizes::Vector{<:Integer})

   appo = 0
   for k in 1:length(inds)
      appo += (inds[k]-1) * prod(sizes[k+1:end])
   end
   return appo+1
end


reshapeTama (generic function with 2 methods)

In [178]:
g1 = Index(4, "combined i',i")
g2 = Index(4, "combined j',j")
combO = ITensor(ComplexF64, g1, g2)
for iprime in 1:2
   for istd in 1:2
      for jprime in 1:2
         for jstd in 1:2
            combO[g1=>reshapeTama([iprime,istd], [2,2]), g2=>reshapeTama([jprime,jstd], [2,2])] = twoSiteOp[i=>istd, i'=>iprime, j=>jstd, j'=>jprime]
         end
      end
   end
end


We have obtained a rank-2 tensor `combO` with indices `g1` $= (i',i)$ and `g2` $ = (j'j)$. We now perform an SVD of `combO`


In [179]:
U,S,V = svd(combO, [g1])

ITensors.TruncSVD(ITensor ord=2
Dim 1: (dim=4|id=448|"combinedi',i")
Dim 2: (dim=4|id=648|"Link,u")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 4×4
                 0.0 + 0.0im  …  -1.0087576610494613e-16 + 0.0im
 -0.7071067811865476 + 0.0im         -0.7071067811865476 + 0.0im
 -0.7071067811865476 + 0.0im          0.7071067811865475 + 0.0im
                 0.0 + 0.0im                         0.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=4|id=648|"Link,u")
Dim 2: (dim=4|id=620|"Link,v")
NDTensors.Diag{Float64, Vector{Float64}}
 4×4
 2.0  0.0                     0.0  0.0
 0.0  1.7730231858351657e-16  0.0  0.0
 0.0  0.0                     0.0  0.0
 0.0  0.0                     0.0  0.0
, ITensor ord=2
Dim 1: (dim=4|id=608|"combinedj',j")
Dim 2: (dim=4|id=620|"Link,v")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 4×4
 -0.7071067811865476 + 0.0im  …   0.0 + 0.0im  0.0 + 0.0im
                 0.0 + 0.0im     -1.0 + 0.0im  0.0 + 0.0im
                -0.0 + 0.0im      0.0 + 0.0im  1.0 +

In [181]:
#Check that the SVD is correct
combO ≈ U * S *dag(V)

true

We now have everything we need. We have a degree of freedom: we can decide to absorb the singular values `S` in `U` or in `V`. We choose to absorb it in `V`. Mind the fact that, for operators, the meaning of the orthogonality center is a bit obscure...

In [183]:
M1appo = U*S
M2appo = dag(V)
M1appo * M2appo ≈ combO

true

In [193]:
prova = commoninds(M1appo, M2appo)
dim(prova[1])

4

We now need to reshape the rank-2 tensors (one combined physical index and one bond index, since they are both extremal, in this case) `M1appo` and `M2appo` into rank-3 tensors

In [199]:
#The function gets a position in the vectorized version of the tensor and an array of sizes and returns the corresponding array of indices
function invReshapeTama(pos::Integer, sizes::Vector{<:Integer})
   inds = zeros(Int, length(sizes))
   for k in 1:length(sizes)
      inds[k] = div(pos-1, prod(sizes[k+1:end])) + 1
      pos -= (inds[k]-1) * prod(sizes[k+1:end])
   end
   return inds
end
#Debug: test that the two functions are inverses of each other

invReshapeTama(reshapeTama([1,1], [2,2]), [2,2])


2-element Vector{Int64}:
 1
 1

In [196]:
bondInx = commoninds(M1appo, M2appo)[1]
bDim = dim(bondInx)
M1 = ITensor(ComplexF64, i', i,bondInx)
M2 = ITensor(ComplexF64, j', j,bondInx)
for iprime in 1:2
   for istd in 1:2
      for bond in 1:bDim
         M1[i'=>iprime, i=>istd, bondInx=>bond] = M1appo[g1=>reshapeTama([iprime,istd], [2,2]), bondInx=>bond]
      end
   end
end
for jprime in 1:2
   for jstd in 1:2
      for bond in 1:bDim
         M2[j'=>jprime, j=>jstd, bondInx=>bond] = M2appo[g2=>reshapeTama([jprime,jstd], [2,2]), bondInx=>bond]
      end
   end
end

In [202]:
M1 * M2 ≈ pauliOpX1 * pauliOpZ2

true

The construction of the MPO representation of an operator we performed here above can be extended to operators acting on larger systems. The amount of work and code required will grow quite consistently with the number of sites (new auxiliary indices, more difficult visualization of intermediate results etc), so we limit ourselves to consider this two-site system.

We conclude this part by exploring the other relevant operation, i.e. MPO representation of the sum of operators. We therefore introduce a new operator

In [204]:
twoSiteOp2 = pauliOpZ1 * pauliOpZ2
appo2 = c2 * twoSiteOp2 * c1
matrix(appo2)

4×4 Matrix{ComplexF64}:
 1.0+0.0im   0.0+0.0im   0.0+0.0im  0.0+0.0im
 0.0+0.0im  -1.0+0.0im   0.0+0.0im  0.0+0.0im
 0.0+0.0im   0.0+0.0im  -1.0+0.0im  0.0+0.0im
 0.0+0.0im   0.0+0.0im   0.0+0.0im  1.0+0.0im

We construct the associated "MPS"
$$
O2  = 
\sum_{\substack{i,j \\ i',j'}}  O2_{(i',i),(j',j)} \ket{(i',i)} \otimes \ket{(j'j)}
$$

In [205]:
combO2 = ITensor(ComplexF64, g1, g2)
for iprime in 1:2
   for istd in 1:2
      for jprime in 1:2
         for jstd in 1:2
            combO2[g1=>reshapeTama([iprime,istd], [2,2]), g2=>reshapeTama([jprime,jstd], [2,2])] = twoSiteOp2[i=>istd, i'=>iprime, j=>jstd, j'=>jprime]
         end
      end
   end
end


We can define the corresponding MPO as we did for the first operator:

In [246]:
U2,S2,V2 = svd(combO2, [g1])
M12Appo = U2*S2
M22Appo = dag(V2)
println("check SVD: " * string(M12Appo * M22Appo ≈ combO2))

bondInx = commoninds(M12Appo, M22Appo)[1]
bDim = dim(bondInx)
M12 = ITensor(ComplexF64, i', i,bondInx)
M22 = ITensor(ComplexF64, j', j,bondInx)
for iprime in 1:2
   for istd in 1:2
      for bond in 1:bDim
         M12[i'=>iprime, i=>istd, bondInx=>bond] = M12Appo[g1=>reshapeTama([iprime,istd], [2,2]), bondInx=>bond]
      end
   end
end
for jprime in 1:2
   for jstd in 1:2
      for bond in 1:bDim
         M22[j'=>jprime, j=>jstd, bondInx=>bond] = M22Appo[g2=>reshapeTama([jprime,jstd], [2,2]), bondInx=>bond]
      end
   end
end

println("check MPO for O2: " * string(M12 * M22 ≈ pauliOpZ1 * pauliOpZ2))


check SVD: true
check MPO for O2: true


Now we have:

In [215]:
@show combO

combO = ITensor ord=2
Dim 1: (dim=4|id=448|"combinedi',i")
Dim 2: (dim=4|id=608|"combinedj',j")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 4×4
 0.0 + 0.0im  0.0 + 0.0im  0.0 + 0.0im   0.0 + 0.0im
 1.0 + 0.0im  0.0 + 0.0im  0.0 + 0.0im  -1.0 + 0.0im
 1.0 + 0.0im  0.0 + 0.0im  0.0 + 0.0im  -1.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im  0.0 + 0.0im   0.0 + 0.0im



ITensor ord=2 (dim=4|id=448|"combinedi',i") (dim=4|id=608|"combinedj',j")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

and

In [216]:
@show combO2

combO2 = ITensor ord=2
Dim 1: (dim=4|id=448|"combinedi',i")
Dim 2: (dim=4|id=608|"combinedj',j")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 4×4
  1.0 + 0.0im  0.0 + 0.0im  0.0 + 0.0im  -1.0 + 0.0im
  0.0 + 0.0im  0.0 + 0.0im  0.0 + 0.0im   0.0 + 0.0im
  0.0 + 0.0im  0.0 + 0.0im  0.0 + 0.0im   0.0 + 0.0im
 -1.0 + 0.0im  0.0 + 0.0im  0.0 + 0.0im   1.0 + 0.0im



ITensor ord=2 (dim=4|id=448|"combinedi',i") (dim=4|id=608|"combinedj',j")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

Given this representation of the two operators, we can perform the sum of the operators simply as matrix sum:

In [217]:
combSum = combO + combO2
matrix(combSum)

4×4 Matrix{ComplexF64}:
  1.0+0.0im  0.0+0.0im  0.0+0.0im  -1.0+0.0im
  1.0+0.0im  0.0+0.0im  0.0+0.0im  -1.0+0.0im
  1.0+0.0im  0.0+0.0im  0.0+0.0im  -1.0+0.0im
 -1.0+0.0im  0.0+0.0im  0.0+0.0im   1.0+0.0im

and then proceed with the construction of the corresponding MPO representation:

In [220]:
USum,SSum,VSum = svd(combSum, [g1])
M1SumAppo = USum*SSum
M2SumAppo = dag(VSum)
M1SumAppo * M2SumAppo ≈ combSum

true

In [221]:
bondInx = commoninds(M1SumAppo, M2SumAppo)[1]
bDim = dim(bondInx)
M1Sum = ITensor(ComplexF64, i', i,bondInx)
M2Sum = ITensor(ComplexF64, j', j,bondInx)
for iprime in 1:2
   for istd in 1:2
      for bond in 1:bDim
         M1Sum[i'=>iprime, i=>istd, bondInx=>bond] = M1SumAppo[g1=>reshapeTama([iprime,istd], [2,2]), bondInx=>bond]
      end
   end
end
for jprime in 1:2
   for jstd in 1:2
      for bond in 1:bDim
         M2Sum[j'=>jprime, j=>jstd, bondInx=>bond] = M2SumAppo[g2=>reshapeTama([jprime,jstd], [2,2]), bondInx=>bond]
      end
   end
end

Test:

In [222]:
M1Sum * M2Sum ≈ pauliOpX1 * pauliOpZ2 + pauliOpZ1 * pauliOpZ2

true

What we should never even dream to do is to define the MPO matrices as the sum of the individual tensors of the addenda! In fact:

In [256]:
l1 = findindex(M1,"Link")
l2 = findindex(M12,"Link")
M1bad = delta(l1,l2)* M12 + M1
l1 = findindex(M2,"Link")
l2 = findindex(M22,"Link")
M2bad = delta(l1,l2)* M22 + M2

ITensor ord=3 (dim=4|id=620|"Link,v") (dim=2|id=215|"Spin_1/2,j,site")' (dim=2|id=215|"Spin_1/2,j,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [257]:
M1bad * M2bad ≈ pauliOpX1 * pauliOpZ2 + pauliOpZ1 * pauliOpZ2

false

Keep this in mind!

## Apply MPO to MPS

In [258]:
#We define the basis states of the first spin
#Another way to create the up state
up1 = ITensor(ComplexF64, ComplexF64.([1,0]),i)
@show up1
down1 = noprime(pauliOpX1 * up1)
@show down1

up1 = ITensor ord=1
Dim 1: (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
 1.0 + 0.0im
 0.0 + 0.0im

down1 = ITensor ord=1
Dim 1: (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
 0.0 + 0.0im
 1.0 + 0.0im



ITensor ord=1 (dim=2|id=821|"Spin_1/2,i,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [259]:
#We define the basis states of the second spin
#Another way to create the up state
up2 = ITensor(ComplexF64, ComplexF64.([1,0]),j)
@show up2
down2 = noprime(pauliOpX2 * up2)
@show down2

up2 = ITensor ord=1
Dim 1: (dim=2|id=215|"Spin_1/2,j,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
 1.0 + 0.0im
 0.0 + 0.0im

down2 = ITensor ord=1
Dim 1: (dim=2|id=215|"Spin_1/2,j,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
 0.0 + 0.0im
 1.0 + 0.0im



ITensor ord=1 (dim=2|id=215|"Spin_1/2,j,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

We now want to produce the Bell state 
$$
\Phi^+ = \frac{\ket{00} + \ket{11}}{\sqrt{2}}
$$

In [267]:
#Define state \Phi^+
bellPlus = 1/sqrt(2) * (down1 * down2 + up1 * up2)
matrix(bellPlus)


2×2 Matrix{ComplexF64}:
 0.707107+0.0im       0.0+0.0im
      0.0+0.0im  0.707107+0.0im

The action of the operator $O$ on the Bell plus state is:

In [284]:
println(matrix(noprime(twoSiteOp * bellPlus) -  1/sqrt(2) * (-up1 * down2 + down1 * up2)))
outState =  1/sqrt(2) * (-up1 * down2 + down1 * up2)

ComplexF64[0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im]


ITensor ord=2 (dim=2|id=821|"Spin_1/2,i,site") (dim=2|id=215|"Spin_1/2,j,site")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

We now write the MPS representatio of $\ket{\Phi^+}$

In [275]:
#Performing the SVD of the bell state by putting the index i in U  and the index j in V
U,S,V = svd(bellPlus, i)
#orthogonality center at site 1
A1 = U*S
A2 =  dag(V)
A1*A2 == bellPlus

true

We now apply the MPO representation of $O$ to the MPS representation of $\ket{\Phi^+}$

In [313]:
resO = noprime(M1 * M2 * A1*A2)
println("check: " * string(resO ≈ outState))
matrix(outState)


check: true


2×2 Matrix{ComplexF64}:
      0.0+0.0im  -0.707107+0.0im
 0.707107+0.0im        0.0+0.0im

Great!

### Exercise: do the same check with the operator `O2` and the operator `O+O2`